In [1]:
import numpy as np
import pandas as pd

# 2. Creación de datos sintéticos (5 buckets) con métricas AS-IS completas
np.random.seed(42)
n_buckets = 5

monto_ofertado_as_is = np.array([10000, 15000, 8000, 20000, 12000], dtype=float)
efectividad_as_is = np.array([0.25, 0.20, 0.30, 0.15, 0.22], dtype=float)
tasa_as_is = np.array([0.15, 0.18, 0.12, 0.20, 0.16], dtype=float)

df_resultados = pd.DataFrame({
    "grupo": [f"Bucket_{i+1}" for i in range(n_buckets)],
    "recomendacion": ["Mantener", "Subir", "Bajar", "Revisar", "Optimizar"],
    "tasa_as_is": tasa_as_is,
    "tea_max": [0.22, 0.25, 0.18, 0.28, 0.22],
    "tea_min": [0.10, 0.12, 0.08, 0.15, 0.10],
    "rango": [0.12, 0.13, 0.10, 0.13, 0.12],
    "efectividad_as_is": efectividad_as_is,
    "elasticidad": [-1.5, -1.8, -1.2, -2.0, -1.6],
    "elasticidad_pesimista": [-2.2, -2.5, -1.8, -2.8, -2.3],
    "elasticidad_optimista": [-0.8, -1.0, -0.6, -1.2, -0.9],
    "monto_ofertado_as_is": monto_ofertado_as_is,
    "monto_desembolsado_as_is": monto_ofertado_as_is * efectividad_as_is,
    "ingreso_financiero_as_is": monto_ofertado_as_is * efectividad_as_is * tasa_as_is
})

In [2]:
# 2. Definición de la función de efectividad
def nueva_efectividad(efectividad_as_is, beta_elasticidad, tasa_as_is, variacion_tasa):
    return efectividad_as_is * (((tasa_as_is + variacion_tasa) / tasa_as_is) ** beta_elasticidad)

# 3. Simulación de escenarios de variación de tasas (deltas en puntos base)
deltas_pb = np.arange(-0.025, 0.0251, 0.005)
res = []

for d in deltas_pb:
    # Cálculo de distancia fuera de límites
    df_resultados['distancia_fuera'] = np.maximum(
        0, (df_resultados['tasa_as_is'] + d) - df_resultados['tea_max']
    ) + np.maximum(
        0, df_resultados['tea_min'] - (df_resultados['tasa_as_is'] + d)
    )

    df_resultados['peso'] = np.minimum(1, df_resultados['distancia_fuera'] / df_resultados["rango"])
    
    # Selección de elasticidad según el escenario (positivo o negativo)
    if d >= 0:
        beta_escenario = df_resultados['elasticidad_pesimista']
    else:
        beta_escenario = df_resultados['elasticidad_optimista']
        
    df_resultados['beta_efectiva'] = df_resultados['elasticidad'] + (beta_escenario - df_resultados['elasticidad']) * df_resultados['peso']

    delta_test = d

    # Cálculos TO-BE
    efectividad_to_be = nueva_efectividad(
        efectividad_as_is=df_resultados["efectividad_as_is"],
        beta_elasticidad=df_resultados["beta_efectiva"],
        tasa_as_is=df_resultados["tasa_as_is"],
        variacion_tasa=delta_test
    )
    
    monto_ofertado_to_be = df_resultados['monto_ofertado_as_is'] # Asumiendo la oferta base constante
    monto_desembolsado_to_be = monto_ofertado_to_be * efectividad_to_be
    tasa_to_be = df_resultados["tasa_as_is"] + delta_test
    ingreso_financiero_to_be = monto_ofertado_to_be * efectividad_to_be * tasa_to_be

    # Cálculo de Deltas (To-Be menos As-Is)
    delta_monto_ofertado = monto_ofertado_to_be - df_resultados['monto_ofertado_as_is']
    delta_monto_desembolsado = monto_desembolsado_to_be - df_resultados['monto_desembolsado_as_is']
    delta_efectividad = efectividad_to_be - df_resultados['efectividad_as_is']
    delta_tasa = tasa_to_be - df_resultados['tasa_as_is']
    delta_ingreso_financiero = ingreso_financiero_to_be - df_resultados['ingreso_financiero_as_is']

    # Construcción del dataframe temporal por escenario
    tmp = pd.DataFrame({
        "grupo": df_resultados["grupo"],
        "recomendacion": df_resultados['recomendacion'],
        "ajuste_pb": int(round(d * 10000)),
        "delta_test": delta_test,
        
        # Métricas AS-IS
        "monto_ofertado_as_is": df_resultados['monto_ofertado_as_is'],
        "monto_desembolsado_as_is": df_resultados['monto_desembolsado_as_is'],
        "efectividad_as_is": df_resultados['efectividad_as_is'],
        "tasa_as_is": df_resultados['tasa_as_is'],
        "ingreso_financiero_as_is": df_resultados['ingreso_financiero_as_is'],
        
        # Métricas TO-BE
        "monto_ofertado_to_be": monto_ofertado_to_be,
        "monto_desembolsado_to_be": monto_desembolsado_to_be,
        "efectividad_to_be": efectividad_to_be,
        "tasa_to_be": tasa_to_be,
        "ingreso_financiero_to_be": ingreso_financiero_to_be,
        
        # Deltas (To-Be - As-Is)
        "delta_monto_ofertado": delta_monto_ofertado,
        "delta_monto_desembolsado": delta_monto_desembolsado,
        "delta_efectividad": delta_efectividad,
        "delta_tasa": delta_tasa,
        "delta_ingreso_financiero": delta_ingreso_financiero,
        
        # Variables de control e intermedias
        "beta_efectivo": df_resultados["beta_efectiva"],
        "distancia_fuera": df_resultados['distancia_fuera'],
        "peso": df_resultados['peso'],
        "elasticidad_pesimista": df_resultados['elasticidad_pesimista'],
        "elasticidad_optimista": df_resultados["elasticidad_optimista"]
    })

    res.append(tmp)

# 4. Consolidación final de todos los escenarios
df_escenarios = pd.concat(res, ignore_index=True)


In [3]:
df_escenarios

,grupo,recomendacion,ajuste_pb,delta_test,monto_ofertado_as_is,monto_desembolsado_as_is,efectividad_as_is,tasa_as_is,ingreso_financiero_as_is,monto_ofertado_to_be,...,delta_monto_ofertado,delta_monto_desembolsado,delta_efectividad,delta_tasa,delta_ingreso_financiero,beta_efectivo,distancia_fuera,peso,elasticidad_pesimista,elasticidad_optimista
0,Bucket_1,Mantener,-250,-2.500000e-02,10000.0,2500.0,0.25,0.15,375.0,10000.0,...,0.0,786.335345,0.078634,-0.025,35.791918,-1.5,0.0,0.0,-2.2,-0.8
1,Bucket_2,Subir,-250,-2.500000e-02,15000.0,3000.0,0.20,0.18,540.0,15000.0,...,0.0,926.582324,0.061772,-0.025,68.620260,-1.8,0.0,0.0,-2.5,-1.0
2,Bucket_3,Bajar,-250,-2.500000e-02,8000.0,2400.0,0.30,0.12,288.0,8000.0,...,0.0,776.584487,0.097073,-0.025,13.775526,-1.2,0.0,0.0,-1.8,-0.6
3,Bucket_4,Revisar,-250,-2.500000e-02,20000.0,3000.0,0.15,0.20,600.0,20000.0,...,0.0,918.367347,0.045918,-0.025,85.714286,-2.0,0.0,0.0,-2.8,-1.2
4,Bucket_5,Optimizar,-250,-2.500000e-02,12000.0,2640.0,0.22,0.16,422.4,12000.0,...,0.0,824.669952,0.068722,-0.025,45.330443,-1.6,0.0,0.0,-2.3,-0.9
5,Bucket_1,Mantener,-200,-2.000000e-02,10000.0,2500.0,0.25,0.15,375.0,10000.0,...,0.0,598.573974,0.059857,-0.020,27.814617,-1.5,0.0,0.0,-2.2,-0.8
6,Bucket_2,Subir,-200,-2.000000e-02,15000.0,3000.0,0.20,0.18,540.0,15000.0,...,0.0,708.478753,0.047232,-0.020,53.356600,-1.8,0.0,0.0,-2.5,-1.0
7,Bucket_3,Bajar,-200,-2.000000e-02,8000.0,2400.0,0.30,0.12,288.0,8000.0,...,0.0,586.955393,0.073369,-0.020,10.695539,-1.2,0.0,0.0,-1.8,-0.6
8,Bucket_4,Revisar,-200,-2.000000e-02,20000.0,3000.0,0.15,0.20,600.0,20000.0,...,0.0,703.703704,0.035185,-0.020,66.666667,-2.0,0.0,0.0,-2.8,-1.2
9,Bucket_5,Optimizar,-200,-2.000000e-02,12000.0,2640.0,0.22,0.16,422.4,12000.0,...,0.0,628.820268,0.052402,-0.020,35.234837,-1.6,0.0,0.0,-2.3,-0.9


In [4]:
# Seleccionamos los escenarios que máximizen el delta de ingreso financiero (if to be - if as is)
escenarios_optimos = (
    df_escenarios
    .sort_values(["grupo", "delta_ingreso_financiero"])
    .groupby("grupo", as_index=False)
    .last()
)


In [5]:
escenarios_optimos

,grupo,recomendacion,ajuste_pb,delta_test,monto_ofertado_as_is,monto_desembolsado_as_is,efectividad_as_is,tasa_as_is,ingreso_financiero_as_is,monto_ofertado_to_be,...,delta_monto_ofertado,delta_monto_desembolsado,delta_efectividad,delta_tasa,delta_ingreso_financiero,beta_efectivo,distancia_fuera,peso,elasticidad_pesimista,elasticidad_optimista
0,Bucket_1,Mantener,-250,-0.025,10000.0,2500.0,0.25,0.15,375.0,10000.0,...,0.0,786.335345,0.078634,-0.025,35.791918,-1.5,0.0,0.0,-2.2,-0.8
1,Bucket_2,Subir,-250,-0.025,15000.0,3000.0,0.20,0.18,540.0,15000.0,...,0.0,926.582324,0.061772,-0.025,68.620260,-1.8,0.0,0.0,-2.5,-1.0
2,Bucket_3,Bajar,-250,-0.025,8000.0,2400.0,0.30,0.12,288.0,8000.0,...,0.0,776.584487,0.097073,-0.025,13.775526,-1.2,0.0,0.0,-1.8,-0.6
3,Bucket_4,Revisar,-250,-0.025,20000.0,3000.0,0.15,0.20,600.0,20000.0,...,0.0,918.367347,0.045918,-0.025,85.714286,-2.0,0.0,0.0,-2.8,-1.2
4,Bucket_5,Optimizar,-250,-0.025,12000.0,2640.0,0.22,0.16,422.4,12000.0,...,0.0,824.669952,0.068722,-0.025,45.330443,-1.6,0.0,0.0,-2.3,-0.9
